# Phase 9 & 10 — Train Baseline & Small CNN

Here we train two models on our 100-image subset:
1. **Baseline Model:** Logistic Regression on flattened pixels.
2. **Small CNN:** A simple 2-layer convolutional network.

In [1]:
# ── Step 0: Imports ───────────────────────────────────────────────────────────
import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO

DATA_CACHE_DIR = '../data/'
SPLIT_MANIFEST_PATH = '../data/processed/split_manifest.csv'
MODEL_DIR = '../models/'
RESULTS_DIR = '../results/'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
# ── Step 1: Load Data & Prepare Splits ────────────────────────────────────────
manifest = pd.read_csv(SPLIT_MANIFEST_PATH)
info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])
dataset = DataClass(split='train', download=False, root=DATA_CACHE_DIR)
all_images = dataset.imgs
all_labels = dataset.labels.squeeze()

train_idx = manifest[manifest['split'] == 'train']['original_index'].values
val_idx = manifest[manifest['split'] == 'val']['original_index'].values

X_train_raw, y_train = all_images[train_idx], all_labels[train_idx]
X_val_raw, y_val = all_images[val_idx], all_labels[val_idx]

print(f"Training set: {X_train_raw.shape}, {y_train.shape}")
print(f"Validation set: {X_val_raw.shape}, {y_val.shape}")

Training set: (70, 28, 28), (70,)
Validation set: (15, 28, 28), (15,)


In [3]:
# ── Step 2: Phase 9 - Train Baseline (Logistic Regression) ────────────────────
start_time = time.time()

# Flatten images: (N, 28, 28) -> (N, 784)
X_train_flat = X_train_raw.reshape((X_train_raw.shape[0], -1)) / 255.0
X_val_flat = X_val_raw.reshape((X_val_raw.shape[0], -1)) / 255.0

clf = LogisticRegression(random_state=SEED, max_iter=1000)
clf.fit(X_train_flat, y_train)

train_acc = accuracy_score(y_train, clf.predict(X_train_flat))
val_acc = accuracy_score(y_val, clf.predict(X_val_flat))
runtime = time.time() - start_time

baseline_metrics = {
    "model_name": "LogisticRegression_Baseline",
    "hyperparameters": {"max_iter": 1000, "solver": "lbfgs"},
    "train_metric": train_acc,
    "validation_metric": val_acc,
    "runtime_seconds": round(runtime, 2),
    "seed": SEED
}

with open(os.path.join(RESULTS_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(baseline_metrics, f, indent=4)

print("\n=== Baseline Results ===")
print(f"Train Accuracy : {train_acc:.4f}")
print(f"Val Accuracy   : {val_acc:.4f}")
print(f"Runtime        : {runtime:.2f}s")
print("Metrics saved to baseline_metrics.json")


=== Baseline Results ===
Train Accuracy : 1.0000
Val Accuracy   : 0.8000
Runtime        : 0.02s
Metrics saved to baseline_metrics.json


In [4]:
# ── Step 3: PyTorch Dataset & Dataloaders for CNN ─────────────────────────────
class MedMNISTSubset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        from PIL import Image
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor([label], dtype=torch.float32)

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(degrees=10),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_ds = MedMNISTSubset(X_train_raw, y_train, transform=transform_train)
val_ds = MedMNISTSubset(X_val_raw, y_val, transform=transform_eval)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

In [5]:
# ── Step 4: Define Small CNN Architecture ─────────────────────────────────────
class TinyCNN(nn.Module):
    def __init__(self):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # Binary output (logits)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = TinyCNN()

In [6]:
# ── Step 5: Phase 10 - Train CNN ──────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 20

start_time = time.time()
print("\n=== Training Small CNN ===")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    train_acc = correct_train / total_train
    
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
    val_acc = correct_val / total_val
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {train_loss/total_train:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

runtime = time.time() - start_time

cnn_metrics = {
    "model_name": "TinyCNN",
    "hyperparameters": {"epochs": EPOCHS, "batch_size": 16, "lr": 0.001},
    "train_metric": train_acc,
    "validation_metric": val_acc,
    "runtime_seconds": round(runtime, 2),
    "seed": SEED
}

with open(os.path.join(RESULTS_DIR, 'cnn_metrics.json'), 'w') as f:
    json.dump(cnn_metrics, f, indent=4)

# Save Model Checkpoint
MODEL_PATH = os.path.join(MODEL_DIR, 'pneumonia_classifier_v1.pt')
torch.save(model.state_dict(), MODEL_PATH)

print(f"\n✅ CNN Training Complete! Runtime: {runtime:.2f}s")
print(f"✅ Model saved to {MODEL_PATH}")


=== Training Small CNN ===


Epoch [1/20] Loss: 0.6939 | Train Acc: 0.5000 | Val Acc: 0.4667
Epoch [2/20] Loss: 0.6847 | Train Acc: 0.5000 | Val Acc: 0.4667
Epoch [3/20] Loss: 0.6663 | Train Acc: 0.5000 | Val Acc: 0.4667
Epoch [4/20] Loss: 0.6462 | Train Acc: 0.5000 | Val Acc: 0.4667
Epoch [5/20] Loss: 0.6106 | Train Acc: 0.5286 | Val Acc: 0.7333
Epoch [6/20] Loss: 0.5661 | Train Acc: 0.8143 | Val Acc: 0.7333
Epoch [7/20] Loss: 0.5187 | Train Acc: 0.9286 | Val Acc: 0.8667
Epoch [8/20] Loss: 0.4655 | Train Acc: 0.8857 | Val Acc: 0.7333
Epoch [9/20] Loss: 0.3960 | Train Acc: 0.9286 | Val Acc: 0.8000


Epoch [10/20] Loss: 0.3429 | Train Acc: 0.9286 | Val Acc: 0.8000
Epoch [11/20] Loss: 0.2903 | Train Acc: 0.9286 | Val Acc: 0.8667
Epoch [12/20] Loss: 0.2557 | Train Acc: 0.9429 | Val Acc: 0.8000
Epoch [13/20] Loss: 0.2229 | Train Acc: 0.9286 | Val Acc: 0.8000
Epoch [14/20] Loss: 0.1762 | Train Acc: 0.9714 | Val Acc: 0.9333
Epoch [15/20] Loss: 0.1576 | Train Acc: 0.9714 | Val Acc: 1.0000
Epoch [16/20] Loss: 0.1579 | Train Acc: 0.9429 | Val Acc: 0.9333
Epoch [17/20] Loss: 0.1183 | Train Acc: 0.9857 | Val Acc: 0.9333
Epoch [18/20] Loss: 0.1270 | Train Acc: 0.9571 | Val Acc: 0.9333
Epoch [19/20] Loss: 0.1162 | Train Acc: 0.9571 | Val Acc: 0.9333


Epoch [20/20] Loss: 0.0885 | Train Acc: 0.9857 | Val Acc: 0.9333

✅ CNN Training Complete! Runtime: 1.21s
✅ Model saved to ../models/pneumonia_classifier_v1.pt
